I want this file to take the large .xlsx file and process it all so we only have usable data for each of the desired streams. Will start by reading and understanding the columns, selecting the columns of interest and then processing each river into its own seperate .csv file. 

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import xarray
import pandas as pd
from pathlib import Path


In [7]:
###Read .xlsx file and select columns of interest
XLSX = Path("../../data/Salmon Data/Johnstone Strait and Strait of Georgia NuSEDS_20260601.xlsx")

xl = pd.ExcelFile(XLSX)
xl.sheet_names

['JSTSOG Data', 'Sheet2', 'metadata']

In [9]:
##open data sheet of interest and parse columns
df = xl.parse("JSTSOG Data")
df.shape
df.columns.tolist()
df.info()
df.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74487 entries, 0 to 74486
Data columns (total 51 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   AREA                       74487 non-null  object        
 1   WATERBODY                  74487 non-null  object        
 2   GAZETTED_NAME              51948 non-null  object        
 3   LOCAL_NAME_1               48310 non-null  object        
 4   LOCAL_NAME_2               21428 non-null  object        
 5   ANALYSIS_YR                74487 non-null  int64         
 6   SPECIES                    74487 non-null  object        
 7   NATURAL_ADULT_SPAWNERS     8072 non-null   float64       
 8   NATURAL_JACK_SPAWNERS      3538 non-null   float64       
 9   NATURAL_SPAWNERS_TOTAL     6177 non-null   float64       
 10  ADULT_BROODSTOCK_REMOVALS  4022 non-null   float64       
 11  JACK_BROODSTOCK_REMOVALS   3057 non-null   float64       
 12  TOTA

AREA                                 object
WATERBODY                            object
GAZETTED_NAME                        object
LOCAL_NAME_1                         object
LOCAL_NAME_2                         object
ANALYSIS_YR                           int64
SPECIES                              object
NATURAL_ADULT_SPAWNERS              float64
NATURAL_JACK_SPAWNERS               float64
NATURAL_SPAWNERS_TOTAL              float64
ADULT_BROODSTOCK_REMOVALS           float64
JACK_BROODSTOCK_REMOVALS            float64
TOTAL_BROODSTOCK_REMOVALS           float64
OTHER_REMOVALS                      float64
TOTAL_RETURN_TO_RIVER               float64
ENUMERATION_METHODS                  object
ADULT_PRESENCE                       object
JACK_PRESENCE                        object
START_DTT                    datetime64[ns]
END_DTT                      datetime64[ns]
NATURAL_ADULT_FEMALES               float64
NATURAL_ADULT_MALES                 float64
EFFECTIVE_FEMALES               

In [14]:
##data of interest
COLUMNS = ["WATERBODY","ANALYSIS_YR","SPECIES","RUN_TYPE","TOTAL_RETURN_TO_RIVER", "START_DTT","STREAM_ARRIVAL_DT_FROM"]

##interests
WB_Interest = ["COWICHAN RIVER","CHEMAINUS RIVER","NANAIMO RIVER","LITTLE QUALICUM RIVER"]
SP_Interest = ["Chinook"]
RT_Interest = ["FALL"]

###Filter data to only include interests
df_filtered = df[df["WATERBODY"].isin(WB_Interest) & df["SPECIES"].isin(SP_Interest) & df["RUN_TYPE"].isin(RT_Interest)]
df_filtered = df_filtered[COLUMNS]

##remove empty variables 
df_filtered = df_filtered.dropna(subset=["TOTAL_RETURN_TO_RIVER"])

##print head
df_filtered.head(10)



,WATERBODY,ANALYSIS_YR,SPECIES,RUN_TYPE,TOTAL_RETURN_TO_RIVER,START_DTT,STREAM_ARRIVAL_DT_FROM
63,COWICHAN RIVER,2025,Chinook,FALL,51209.0,2025-09-08,NaT
175,LITTLE QUALICUM RIVER,2025,Chinook,FALL,7755.0,2025-09-05,NaT
490,CHEMAINUS RIVER,2024,Chinook,FALL,1204.0,2024-10-04,NaT
517,COWICHAN RIVER,2024,Chinook,FALL,42569.0,2024-09-06,NaT
724,LITTLE QUALICUM RIVER,2024,Chinook,FALL,7924.0,2024-08-29,NaT
799,NANAIMO RIVER,2024,Chinook,FALL,7501.0,2024-09-09,NaT
1163,CHEMAINUS RIVER,2023,Chinook,FALL,437.0,2023-10-06,NaT
1179,COWICHAN RIVER,2023,Chinook,FALL,35324.0,2023-09-08,NaT
1303,LITTLE QUALICUM RIVER,2023,Chinook,FALL,7868.0,2023-09-18,NaT
1674,CHEMAINUS RIVER,2022,Chinook,FALL,77.0,2022-09-16,NaT


In [17]:
##process which start time to use
## if the stream arrival date is not available, use the start date. If the stream arrival date is available, use that instead.
df_filtered["START_DTT"] = pd.to_datetime(df_filtered["START_DTT"])
df_filtered["STREAM_ARRIVAL_DT_FROM"] = pd.to_datetime(df_filtered["STREAM_ARRIVAL_DT_FROM"])
##input into new column called "time_return"
df_filtered["time_return"] = df_filtered["STREAM_ARRIVAL_DT_FROM"].combine_first(df_filtered["START_DTT"])  

##remove empty variables 
df_filtered = df_filtered.dropna(subset=["time_return"])

##print head - check
df_filtered.head(100)



,WATERBODY,ANALYSIS_YR,SPECIES,RUN_TYPE,TOTAL_RETURN_TO_RIVER,START_DTT,STREAM_ARRIVAL_DT_FROM,time_return
63,COWICHAN RIVER,2025,Chinook,FALL,51209.0,2025-09-08,NaT,2025-09-08
175,LITTLE QUALICUM RIVER,2025,Chinook,FALL,7755.0,2025-09-05,NaT,2025-09-05
490,CHEMAINUS RIVER,2024,Chinook,FALL,1204.0,2024-10-04,NaT,2024-10-04
517,COWICHAN RIVER,2024,Chinook,FALL,42569.0,2024-09-06,NaT,2024-09-06
724,LITTLE QUALICUM RIVER,2024,Chinook,FALL,7924.0,2024-08-29,NaT,2024-08-29
...,...,...,...,...,...,...,...,...
32557,NANAIMO RIVER,1988,Chinook,FALL,1755.0,1988-09-08,NaT,1988-09-08
33184,CHEMAINUS RIVER,1987,Chinook,FALL,200.0,1987-09-10,1987-10-01,1987-10-01
33563,LITTLE QUALICUM RIVER,1987,Chinook,FALL,375.0,NaT,1987-09-28,1987-09-28
33672,NANAIMO RIVER,1987,Chinook,FALL,618.0,1987-08-19,NaT,1987-08-19


In [18]:
##For one stream, create .csv using all available data 
for stream in WB_Interest:
    df_stream = df_filtered[df_filtered["WATERBODY"] == stream]
    df_stream.to_csv(f"../../data/Salmon Data/{stream}_salmon_data.csv", index=False)
    print(f"Saved {stream} salmon data to CSV file.")

Saved COWICHAN RIVER salmon data to CSV file.
Saved CHEMAINUS RIVER salmon data to CSV file.
Saved NANAIMO RIVER salmon data to CSV file.
Saved LITTLE QUALICUM RIVER salmon data to CSV file.
